## RQ1: Predictive Performance & Key Lending Drivers
#### Group 6 | Francesco Biedermann (FB2709) | Brian Hsu (CH4004) | Phoebe Zhao (PYZ2001) 
#### APAN5205 Applied Machine Learning 2 | Prof. Andrew Assing

This notebook addresses Research Question 1: *"To what extent can mortgage approval outcomes be predicted using applicant, loan, and property characteristics available at the time of application, and which factors appear to play the largest role in these decisions?"*

We train and compare three classification models:
- Logistic Regression (with Lasso and Ridge regularization)
- Decision Tree
- Random Forest (using 5-fold cross-validation for hyperparameter tuning)

Demographic variables are excluded from the predictive models and reserved for the fairness analysis in RQ2.

### Step 1 | Setup and Data Loading
We begin by importing the required libraries and loading the cleaned HMDA dataset produced during data preparation. The dataset contains 433,173 mortgage application records with 32 columns and zero missing values.

In [9]:
# Step 1: Setup and Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, roc_curve, ConfusionMatrixDisplay

# Load cleaned data
df = pd.read_csv('../data/hmda_2024_cleaned.csv')

print('Dataset shape:', df.shape)
print('Approval rate:', round(df['approved'].mean()*100, 2), '%')
print('\nTarget distribution:')
print(df['approved'].value_counts())

Dataset shape: (433173, 32)
Approval rate: 77.36 %

Target distribution:
approved
1    335087
0     98086
Name: count, dtype: int64


### Step 2 | Feature Selection and Preparation
We separate the target variable from the features and apply several preparation steps:

- **Demographic exclusion:** The derived race, ethnicity and sex columns are excluded from the predictive models, consistent with our research design. These variables are reserved for the fairness analysis in RQ2
- **Constant feature removal:** The "reverse_mortgage" column contains only a single value (2) after data cleaning and provides no discriminative information. It is therefore dropped
- **One-hot encoding:** The "state_code" column (54 US states) is one-hot encoded with "drop_first=True" to avoid multicollinearity

In [11]:
# Step 2: Feature Selection and Preparation

# Define target
y = df['approved']

# Drop target, demographic columns (reserved for RQ2), and constant column
X = df.drop(columns=['approved', 'derived_race', 'derived_ethnicity', 'derived_sex', 'reverse_mortgage'])

print('Dropping reverse_mortgage, only one unique value:', df['reverse_mortgage'].unique())

# One-hot encode state_code
X = pd.get_dummies(X, columns=['state_code'], drop_first=True)

print('\nFeature matrix shape:', X.shape)
print('Number of features:', X.shape[1])

Dropping reverse_mortgage, only one unique value: [2]

Feature matrix shape: (433173, 79)
Number of features: 79


### Step 3 | Train-Test Split
We split the data 70/30 into training and test sets, stratifying on the target variable to preserve the approval rate in both partitions. We also draw a stratified subsample of 50,000 rows from the training set for hyperparameter tuning, as running 5-fold cross-validation on 300,000+ rows would be challenging. Once the best hyperparameters are identified, the final models are trained on the full training set.

We also standardize features using "StandardScaler" for use with Logistic Regression. Because Logistic Regression with L1 and L2 regularization applies a penalty to the coefficients, features must be on the same scale for the penalty to be applied uniformly. The scaler is fit on the training data and applied to both training and test sets to prevent data leakage. Decision Trees and Random Forests are not affected by feature scale and are trained on unscaled data.

In [12]:
# Step 3: Train-Test Split and Scaling

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print('Training set:', X_train.shape[0], 'rows')
print('Test set:    ', X_test.shape[0], 'rows')
print('Approval rate - Train:', round(y_train.mean()*100, 2), '%')
print('Approval rate - Test: ', round(y_test.mean()*100, 2), '%')

# Standardize features for Logistic Regression (fit on train, transform both)
scale = StandardScaler()
scale.fit(X_train)
X_train_scaled = scale.transform(X_train)
X_test_scaled = scale.transform(X_test)

print('\nStandardScaler applied for Logistic Regression.')

# Subsample for hyperparameter tuning (full training set is too large for CV)
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train, train_size=50000, random_state=42, stratify=y_train
)

# Also create scaled version of tuning subsample
X_tune_scaled = scale.transform(X_tune)

print('Tuning subsample:', X_tune.shape[0], 'rows')
print('Approval rate - Tuning:', round(y_tune.mean()*100, 2), '%')

Training set: 303221 rows
Test set:     129952 rows
Approval rate - Train: 77.36 %
Approval rate - Test:  77.36 %

StandardScaler applied for Logistic Regression.
Tuning subsample: 50000 rows
Approval rate - Tuning: 77.36 %


### Step 4a | Model 1: Logistic Regression (Ridge)
We train Logistic Regression with L2 (Ridge) regularization on the standardized features, tuning the regularization strength via 5-fold cross-validation. Ridge shrinks all coefficients toward zero without eliminating them, which can improve generalization when many features are moderately important.

In [13]:
# Step 4a: Logistic Regression — L2 (Ridge) on scaled data

logit_l2 = LogisticRegression(max_iter=10000, penalty='l2', random_state=42)

param_grid_l2 = {'C': [0.01, 0.1, 1, 10]}

tune_logit_l2 = GridSearchCV(
    estimator=logit_l2,
    param_grid=param_grid_l2,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)

tune_logit_l2.fit(X_tune_scaled, y_tune)

print('L2 (Ridge) Results:')
print('Best C:', tune_logit_l2.best_params_['C'])
print('Best CV AUC-ROC:', round(tune_logit_l2.best_score_, 4))

# Show all CV results
cv_l2 = pd.DataFrame(tune_logit_l2.cv_results_)
print('\nAll results:')
print(cv_l2[['param_C', 'mean_test_score', 'std_test_score']].to_string(index=False))

L2 (Ridge) Results:
Best C: 1
Best CV AUC-ROC: 0.9999

All results:
 param_C  mean_test_score  std_test_score
    0.01         0.999844        0.000074
    0.10         0.999910        0.000057
    1.00         0.999929        0.000049
   10.00         0.999927        0.000051


### Step 4b | Model 1: Logistic Regression (Lasso)
We train Logistic Regression with L1 (Lasso) regularization on the standardized features. Lasso performs implicit feature selection by shrinking some coefficients to exactly zero, helping identify which variables are most relevant to the lending decision. The "saga" solver is required for L1 regularization.

In [14]:
# Step 4b: Logistic Regression — L1 (Lasso) on scaled data

logit_l1 = LogisticRegression(solver='saga', max_iter=10000, penalty='l1', random_state=42)

param_grid_l1 = {'C': [0.01, 0.1, 1, 10]}

tune_logit_l1 = GridSearchCV(
    estimator=logit_l1,
    param_grid=param_grid_l1,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)

tune_logit_l1.fit(X_tune_scaled, y_tune)

print('L1 (Lasso) Results:')
print('Best C:', tune_logit_l1.best_params_['C'])
print('Best CV AUC-ROC:', round(tune_logit_l1.best_score_, 4))

# Show all CV results
cv_l1 = pd.DataFrame(tune_logit_l1.cv_results_)
print('\nAll results:')
print(cv_l1[['param_C', 'mean_test_score', 'std_test_score']].to_string(index=False))

L1 (Lasso) Results:
Best C: 1
Best CV AUC-ROC: 0.9999

All results:
 param_C  mean_test_score  std_test_score
    0.01         0.999385        0.000150
    0.10         0.999920        0.000055
    1.00         0.999931        0.000048
   10.00         0.999928        0.000048


### Step 4c | Model 1: Logistic Regression (Select Best and Refit)
We compare the best L1 and L2 results and refit the winning model on the full scaled training set.

In [16]:
# Step 4c: Select best LR and refit on full scaled training data

print('L2 Best CV AUC-ROC:', round(tune_logit_l2.best_score_, 4))
print('L1 Best CV AUC-ROC:', round(tune_logit_l1.best_score_, 4))

if tune_logit_l2.best_score_ >= tune_logit_l1.best_score_:
    best_lr = tune_logit_l2.best_estimator_
    print('\nSelected: L2 (Ridge) with C =', tune_logit_l2.best_params_['C'])
else:
    best_lr = tune_logit_l1.best_estimator_
    print('\nSelected: L1 (Lasso) with C =', tune_logit_l1.best_params_['C'])

# Refit on full scaled training data
best_lr.fit(X_train_scaled, y_train)
print('Refitted on full training data (scaled).')

L2 Best CV AUC-ROC: 0.9999
L1 Best CV AUC-ROC: 0.9999

Selected: L1 (Lasso) with C = 1
Refitted on full training data (scaled).


### Step 5 | Model 2: Decision Tree
We train a Decision Tree classifier on the unscaled data, tuning the maximum depth and minimum samples at leaf nodes via 5-fold cross-validation. Decision Trees are not affected by feature scale. They learn axis-aligned decision rules and are highly interpretable but prone to overfitting, which the depth and leaf constraints help mitigate.

In [17]:
# Step 5: Decision Tree with GridSearchCV (unscaled data)

tree = DecisionTreeClassifier(random_state=42)

param_grid = {
    'max_depth': [3, 5, 7, 10, 15],
    'min_samples_leaf': [50, 100, 200, 500]
}

tune_tree = GridSearchCV(
    estimator=tree,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)

tune_tree.fit(X_tune, y_tune)

print('Best parameters:', tune_tree.best_params_)
print('Best CV AUC-ROC:', round(tune_tree.best_score_, 4))

# Show CV results
cv_tree = pd.DataFrame(tune_tree.cv_results_)
cv_tree = cv_tree[['param_max_depth', 'param_min_samples_leaf', 'mean_test_score', 'std_test_score']]
cv_tree = cv_tree.sort_values('mean_test_score', ascending=False)
print('\nTop 10 results:')
print(cv_tree.head(10).to_string(index=False))

Best parameters: {'max_depth': 10, 'min_samples_leaf': 100}
Best CV AUC-ROC: 0.9999

Top 10 results:
 param_max_depth  param_min_samples_leaf  mean_test_score  std_test_score
              15                     100         0.999853        0.000139
              10                     100         0.999853        0.000139
              15                     200         0.999845        0.000080
              10                     200         0.999845        0.000080
               7                     100         0.999842        0.000132
               7                     200         0.999840        0.000086
               5                     500         0.999832        0.000044
               5                     100         0.999831        0.000151
               7                     500         0.999826        0.000042
              10                     500         0.999826        0.000042


### Step 6 | Model 3: Random Forest
We train a Random Forest classifier, which builds an ensemble of decorrelated decision trees and averages their predictions. This reduces the overfitting tendency of individual trees while typically improving predictive accuracy. We tune the number of trees, maximum depth and minimum leaf size via 5-fold cross-validation.

In [8]:
# Step 6: Random Forest with Hyperparameter Tuning

rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [50, 100, 200]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {rf_grid.best_params_}")
print(f"Best CV AUC-ROC: {rf_grid.best_score_:.4f}")

# Show top 10 CV results
rf_cv_results = pd.DataFrame(rf_grid.cv_results_)
rf_cv_results = rf_cv_results[['param_n_estimators', 'param_max_depth', 'param_min_samples_leaf', 'mean_test_score', 'std_test_score']]
rf_cv_results = rf_cv_results.sort_values('mean_test_score', ascending=False)
print(f"\nTop 10 CV results:")
print(rf_cv_results.head(10).to_string(index=False))

rf_best = rf_grid.best_estimator_

Fitting 5 folds for each of 18 candidates, totalling 90 fits

Best parameters: {'max_depth': 20, 'min_samples_leaf': 50, 'n_estimators': 100}
Best CV AUC-ROC: 0.9999

Top 10 CV results:
 param_n_estimators param_max_depth  param_min_samples_leaf  mean_test_score  std_test_score
                100              20                      50         0.999902        0.000030
                200              20                      50         0.999896        0.000037
                200            None                      50         0.999884        0.000030
                100            None                      50         0.999878        0.000034
                200              20                     200         0.999875        0.000028
                100              10                      50         0.999870        0.000046
                100              20                     200         0.999869        0.000020
                200            None                     200         0.